# Home Credit Default Risk - ALL-IN-ONE notebook (one T4 GPU, several models at once, plus TWIN adjustment)

**Set up:** *Runtime -> Change runtime type -> T4 GPU*, then upload `train*.csv`, `test.csv` and `sample_submission.csv` to `/content` (or mount Drive in cell 3). Run the cells in order. **The file to submit is `/content/submission_twins.csv`** (`submission.csv` is the plain blend without the twin stage).

**What it does (everything in this one notebook)**
1. **Features, built once:** cleaning (`DAYS_EMPLOYED` sentinel, `XNA`, outliers), ratio / age / enquiry features, cross-fitted imputation of `EXT_SOURCE_1/2/3` (the data has extra randomly blanked values there), and the **loan-product clusters** from `AMT_CREDIT / AMT_ANNUITY` with in-fold target encoding (default rate ranges from ~1% to ~25% across clusters).
2. **Up to 4 models train at the same time as separate processes** sharing the one T4: GPU trees (`xgb`, `xgb2`, `cat` - CatBoost capped at 40% of GPU memory), the CPU tree `hgb_f`, and linear / neural models fed quantile + one-hot + **spline** features (`lr`, `mlpA`, `mlpB`, `mlp3`).
3. **Blend:** rank-average with weights fitted on out-of-fold predictions (hill-climb from equal weights).
4. **Twin adjustment (the biggest single gain after the clusters):** the same person appears under different IDs. Application dates differ, so every `DAYS_*` field shifts by the same amount and differences such as `DAYS_REGISTRATION - DAYS_BIRTH` stay constant - a fingerprint. Among train pairs the second application defaults **41.5%** of the time if the first defaulted and **3.5%** if not. The direction matters a lot: if a person's **earlier** application defaulted, their later loan defaults ~70% of the time; if the **later** one defaulted, the earlier loan defaults ~23%. A second-stage logistic regression turns the train twins' outcomes split by direction (plus time gap, contract type, change in requested credit/income, twin counts) into an adjustment of the blend; the final stage averages three differently-built second stages (a logistic regression on the core features, a more regularised one on every twin feature, a small gradient-boosted one); it is validated with group cross-validation (a person never straddles train and validation) and gained +0.0033 AUC. About 2.7% of rows have a twin.

Every finished fold is saved, so after a Colab disconnect just re-run the run cell and it resumes. The printed **test-like OOF AUC** (validation re-weighted to resemble the test set) and the twin-stage group-CV gain are the numbers to compare with the leaderboard.

**Honest expectations:** on the CPU models alone the blend scored about 0.774 on the leaderboard; the twin stage added about +0.0016 on an earlier version. The GPU models have not been run on a real T4 by me. Reaching 0.80 is not supported by anything measured on these 32 columns.

In [ ]:
!nvidia-smi -L
!pip -q install catboost
import xgboost, catboost, sklearn
print('xgboost', xgboost.__version__, '| catboost', catboost.__version__, '| sklearn', sklearn.__version__)

In [ ]:
# Data: upload the csv files, or mount Google Drive and point DATA_DIR at the folder that holds them.
import glob, os
if not glob.glob('/content/train*.csv'):
    # from google.colab import drive; drive.mount('/content/drive'); os.environ['DATA_DIR'] = '/content/drive/MyDrive/your_folder'
    from google.colab import files
    files.upload()   # select train*.csv, test.csv, sample_submission.csv
print(glob.glob('/content/*.csv'))

In [ ]:
%%writefile home_credit_all_in_one.py
"""Home Credit Default Risk - ALL-IN-ONE pipeline (features -> GPU/CPU tree models + linear + neural models -> blend -> TWIN ADJUSTMENT).

    python home_credit_colab.py all            # features -> models in parallel -> blend -> submission.csv
    python home_credit_colab.py prep | train <model> | blend      # individual stages

Models: xgb, xgb2 (GPU) | cat (GPU) | lgb (CPU) | hgb, hgb_a, hgb_f (CPU, sklearn trees) | lr, mlpA, mlpB, mlpC, mlp3 (CPU, spline-fed linear / neural models).
Every model runs in its own process (up to MAX_PARALLEL at once), every finished fold is checkpointed in
WORK_DIR/preds, so a Colab disconnect only loses the fold in progress - re-run and it resumes.
"""
import os, sys, re, glob, time, pickle, subprocess, traceback, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata, norm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder, SplineTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")

# ----------------------------- CONFIG (override with environment variables) -----------------------------
SEED         = 42
N_FOLDS      = int(os.environ.get("N_FOLDS", 10))
N_SEEDS      = int(os.environ.get("N_SEEDS", 1))
MODELS       = os.environ.get("MODELS", "xgb,xgb2,cat,hgb_f,lr,mlpA,mlpB,mlp3").split(",")   # up to MAX_PARALLEL run at the same time
MAX_PARALLEL = int(os.environ.get("MAX_PARALLEL", 4))          # models training simultaneously
SMOKE        = os.environ.get("SMOKE", "0") == "1"             # tiny debug run
DATA_DIR     = os.environ.get("DATA_DIR")                      # optional: folder holding the csv files
_default_root = "/content" if os.path.isdir("/content") else ("/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())
WORK_DIR     = os.environ.get("WORK_DIR") or os.path.join(_default_root, "work")   # use a Google-Drive path to survive disconnects
OUT_DIR      = os.environ.get("OUT_DIR") or _default_root
MAX_ROUNDS   = 60 if SMOKE else 10000
if SMOKE: N_FOLDS = 3
THREADS      = int(os.environ.get("HC_THREADS", os.cpu_count() or 2))
CAT_GPU_RAM_PART = float(os.environ.get("CAT_GPU_RAM_PART", 0.4))   # CatBoost would otherwise grab ~95% of the T4's memory
TE_RES, TE_ALPHA = [0.5, 0.1, 0.05], 30.0
TAG          = "_smoke" if SMOKE else ""
CACHE        = os.path.join(WORK_DIR, f"cache{TAG}.pkl")
PRED_DIR     = os.path.join(WORK_DIR, f"preds{TAG}")
LOG_DIR      = os.path.join(WORK_DIR, f"logs{TAG}")

def count_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=20).stdout
        return sum(1 for l in out.splitlines() if l.startswith("GPU"))
    except Exception:
        return 0
N_GPUS = int(os.environ.get("FORCE_GPUS", count_gpus()))

def locate(patterns, exclude=()):
    roots = [DATA_DIR, "/kaggle/input", "/content", "/content/drive/MyDrive", "/mnt/user-data/uploads",
             os.getcwd(), os.path.join(os.getcwd(), "input")]
    for r in roots:
        if not r or not os.path.isdir(r): continue
        for pat in patterns:
            hits = []
            for depth in ("", "*", "*/*", "*/*/*"):          # shallow search only: Drive folders can be huge
                hits += glob.glob(os.path.join(r, depth, pat))
            hits = [h for h in sorted(set(hits)) if not any(x in os.path.basename(h).lower() for x in exclude)]
            if hits: return hits[0]
    raise FileNotFoundError(f"could not find {patterns}; upload the csv files to /content or set DATA_DIR")

# ----------------------------- FEATURES -----------------------------
CATS = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
        "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE"]
def sdiv(a, b): return a / b.replace(0, np.nan)

def build_base(df):
    d = df.copy()
    # --- cleaning
    d["DAYS_EMPLOYED_ANOM"] = (d.DAYS_EMPLOYED == 365243).astype(int)          # sentinel for "not employed / pensioner"
    d.loc[d.DAYS_EMPLOYED == 365243, "DAYS_EMPLOYED"] = np.nan
    d.loc[d.CODE_GENDER == "XNA", "CODE_GENDER"] = "F"
    d.loc[d.AMT_REQ_CREDIT_BUREAU_QRT > 50, "AMT_REQ_CREDIT_BUREAU_QRT"] = np.nan
    d.loc[d.OWN_CAR_AGE >= 60, "OWN_CAR_AGE"] = np.nan
    d["AGE_YEARS"] = -d.DAYS_BIRTH / 365.25
    # --- money ratios
    d["CREDIT_INCOME"] = sdiv(d.AMT_CREDIT, d.AMT_INCOME_TOTAL)
    d["ANNUITY_INCOME"] = sdiv(d.AMT_ANNUITY, d.AMT_INCOME_TOTAL)
    d["CREDIT_TERM"] = sdiv(d.AMT_ANNUITY, d.AMT_CREDIT)
    d["CREDIT_GOODS"] = sdiv(d.AMT_CREDIT, d.AMT_GOODS_PRICE)
    d["GOODS_INCOME"] = sdiv(d.AMT_GOODS_PRICE, d.AMT_INCOME_TOTAL)
    d["CREDIT_MINUS_GOODS"] = d.AMT_CREDIT - d.AMT_GOODS_PRICE
    d["INCOME_PER_PERSON"] = sdiv(d.AMT_INCOME_TOTAL, d.CNT_FAM_MEMBERS)
    d["INCOME_PER_CHILD"] = sdiv(d.AMT_INCOME_TOTAL, d.CNT_CHILDREN + 1)
    d["CHILD_RATIO"] = sdiv(d.CNT_CHILDREN, d.CNT_FAM_MEMBERS)
    d["CREDIT_PER_PERSON"] = sdiv(d.AMT_CREDIT, d.CNT_FAM_MEMBERS)
    d["ANNUITY_PER_PERSON"] = sdiv(d.AMT_ANNUITY, d.CNT_FAM_MEMBERS)
    d["INCOME_AFTER_ANNUITY"] = d.AMT_INCOME_TOTAL - d.AMT_ANNUITY * 12
    d["CREDIT_YEARS_AT_INCOME"] = sdiv(d.AMT_CREDIT, d.AMT_ANNUITY)
    # --- time features
    d["EMPLOYED_BIRTH"] = sdiv(d.DAYS_EMPLOYED, d.DAYS_BIRTH)
    d["REG_BIRTH"] = sdiv(d.DAYS_REGISTRATION, d.DAYS_BIRTH)
    d["ID_BIRTH"] = sdiv(d.DAYS_ID_PUBLISH, d.DAYS_BIRTH)
    d["PHONE_BIRTH"] = sdiv(d.DAYS_LAST_PHONE_CHANGE, d.DAYS_BIRTH)
    d["AGE_AT_EMPLOY_START"] = (d.DAYS_EMPLOYED - d.DAYS_BIRTH) / 365.25
    d["AGE_AT_ID"] = (d.DAYS_ID_PUBLISH - d.DAYS_BIRTH) / 365.25
    d["AGE_AT_REG"] = (d.DAYS_REGISTRATION - d.DAYS_BIRTH) / 365.25
    d["CAR_EMPLOYED"] = d.OWN_CAR_AGE * 365.25 + d.DAYS_EMPLOYED
    d["CAR_AGE_BIRTH"] = sdiv(d.OWN_CAR_AGE * 365.25, -d.DAYS_BIRTH)
    d["INCOME_PER_WORKDAY"] = sdiv(d.AMT_INCOME_TOTAL, -d.DAYS_EMPLOYED)
    d["CREDIT_TO_AGE"] = sdiv(d.AMT_CREDIT, d.AGE_YEARS)
    d["REGION_RATING_POP"] = d.REGION_RATING_CLIENT * d.REGION_POPULATION_RELATIVE
    # --- credit-bureau enquiries
    e = ["AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR"]
    d["ENQ_TOTAL"] = d[e].sum(axis=1, min_count=1)
    d["ENQ_RECENT"] = d[e[:3]].sum(axis=1, min_count=1)
    d["ENQ_NULL"] = d.AMT_REQ_CREDIT_BUREAU_YEAR.isnull().astype(int)
    d = d.replace([np.inf, -np.inf], np.nan)
    for c in CATS:
        d[c] = d[c].astype(object).astype("category").cat.codes.astype("int32")
    return d.drop(columns=["SK_ID_CURR"])

def add_ext_features(d):
    d = d.copy(); E = d[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    d["EXT_MEAN"] = E.mean(axis=1); d["EXT_MIN"] = E.min(axis=1); d["EXT_MAX"] = E.max(axis=1)
    d["EXT_STD"] = E.std(axis=1); d["EXT_COUNT"] = E.notnull().sum(axis=1); d["EXT_PROD"] = E.prod(axis=1, min_count=1)
    wsum = (2 * d.EXT_SOURCE_1.notnull() + 3 * d.EXT_SOURCE_2.notnull() + 4 * d.EXT_SOURCE_3.notnull()).replace(0, np.nan)
    d["EXT_W"] = (d.EXT_SOURCE_1.fillna(0) * 2 + d.EXT_SOURCE_2.fillna(0) * 3 + d.EXT_SOURCE_3.fillna(0) * 4) / wsum
    d["EXT_23"] = d.EXT_SOURCE_2 * d.EXT_SOURCE_3; d["EXT_12"] = d.EXT_SOURCE_1 * d.EXT_SOURCE_2
    d["EXT_13"] = d.EXT_SOURCE_1 * d.EXT_SOURCE_3; d["EXT2_MINUS_3"] = d.EXT_SOURCE_2 - d.EXT_SOURCE_3
    d["EXT_MEAN_PER_AGE"] = sdiv(d.EXT_MEAN, d.AGE_YEARS)
    d["EXT_MEAN_X_TERM"] = d.EXT_MEAN * d.CREDIT_TERM
    d["EXT_X_CREDIT_INCOME"] = d.EXT_MEAN * d.CREDIT_INCOME
    return d

def make_hgb(cls, **kw):
    """sklearn < 1.4 has no max_features -> drop it transparently"""
    try: return cls(**kw)
    except TypeError:
        kw.pop("max_features", None); return cls(**kw)

def impute_ext(base):
    """Cross-fitted regression imputation of EXT_SOURCE_x (uses features only, never the target).
    Observed rows get out-of-fold predictions, missing rows the average of the fold models."""
    out = {}
    for t in ["EXT_SOURCE_1", "EXT_SOURCE_3", "EXT_SOURCE_2"]:
        t0 = time.time(); cols = [c for c in base.columns if c != t]; Xp = base[cols]
        cidx = [Xp.columns.get_loc(c) for c in CATS]
        obs = base[t].notnull().values; io, im = np.where(obs)[0], np.where(~obs)[0]
        pred = np.zeros(len(base))
        for a, b in KFold(3, shuffle=True, random_state=0).split(io):
            m = make_hgb(HistGradientBoostingRegressor, learning_rate=0.1, max_iter=30 if SMOKE else 200, max_leaf_nodes=31,
                         min_samples_leaf=50, categorical_features=cidx, early_stopping=True, n_iter_no_change=15, random_state=0)
            m.fit(Xp.iloc[io[a]], base[t].values[io[a]])
            pred[io[b]] = m.predict(Xp.iloc[io[b]]); pred[im] += m.predict(Xp.iloc[im]) / 3
        ob = base[t].values[obs]
        print(f"[impute] {t}: OOF R2={1 - ((ob - pred[obs]) ** 2).sum() / ((ob - ob.mean()) ** 2).sum():.3f} ({time.time() - t0:.0f}s)", flush=True)
        out[t] = pred
    return out

def add_imputed(d, imp):
    d = d.copy()
    for t, p in imp.items():
        d[f"{t}_PRED"] = p; d[f"{t}_FILL"] = np.where(d[t].isnull(), p, d[t])
    F = d[["EXT_SOURCE_1_FILL", "EXT_SOURCE_2_FILL", "EXT_SOURCE_3_FILL"]]
    d["EXTF_MEAN"] = F.mean(axis=1); d["EXTF_MIN"] = F.min(axis=1); d["EXTF_MAX"] = F.max(axis=1)
    d["EXTF_PROD"] = F.prod(axis=1); d["EXTF_STD"] = F.std(axis=1)
    d["EXTF_W"] = (d.EXT_SOURCE_1_FILL * 2 + d.EXT_SOURCE_2_FILL * 3 + d.EXT_SOURCE_3_FILL * 4) / 9
    d["EXT1_MISS"] = d.EXT_SOURCE_1.isnull().astype(int); d["EXT3_MISS"] = d.EXT_SOURCE_3.isnull().astype(int)
    return d

XALL = KEYS = y = folds = test_ids = P_SAMPLE = TWIN_COLS = None
n = N = 0; TEST_IDX = None; CL_COLS = ["CL_010", "CL_005"]

def prep():
    """Label-free feature engineering (cleaning, ratios, EXT imputation, loan-product clusters) -> cache file."""
    for d_ in (WORK_DIR, PRED_DIR, LOG_DIR): os.makedirs(d_, exist_ok=True)
    p_train = locate(["train*.csv"], exclude=("sample",)); p_test = locate(["test*.csv"], exclude=("sample",))
    try: p_sample = locate(["sample_submission*.csv"])
    except FileNotFoundError: p_sample = None
    print("[data]", p_train, p_test, p_sample, flush=True)
    train = pd.read_csv(p_train); test = pd.read_csv(p_test)
    target = "TARGET" if "TARGET" in train.columns else "TAREGET"
    if SMOKE:
        train = train.sample(30000, random_state=0).reset_index(drop=True); test = test.iloc[:8000].reset_index(drop=True)
    yy = train[target].astype(int).values; nn = len(train)
    full = pd.concat([train.drop(columns=[target]), test], axis=0, ignore_index=True); twin_cols = full[TWIN_COLUMNS].copy()
    print(f"[data] train={train.shape} test={test.shape} positives={yy.mean():.4f}", flush=True)
    t0 = time.time()
    base = build_base(full); imp = impute_ext(base)
    X = add_imputed(add_ext_features(base), imp)
    inv = (full.AMT_CREDIT / full.AMT_ANNUITY).values                 # identifies the (term, rate tier) loan product
    ckey = lambda res: np.round(np.nan_to_num(inv, nan=-9) / res).astype(np.int64)
    X["INV_CREDIT_ANNUITY"] = inv
    for r in (0.1, 0.05, 0.02):
        k = pd.Series(ckey(r)); X[f"CNT_INV_{int(round(r * 100)):03d}"] = k.map(k.value_counts()).values
    keys = {f"TE_INV_R{int(round(r * 100)):03d}": ckey(r) for r in TE_RES}
    X["CL_010"] = ckey(0.1); X["CL_005"] = ckey(0.05)
    for c in X.columns:
        if X[c].dtype == np.float64: X[c] = X[c].astype(np.float32)
    X = X.reset_index(drop=True)
    fl = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(nn), yy))
    pickle.dump(dict(XALL=X, KEYS=keys, y=yy, n=nn, N=len(full), test_ids=test["SK_ID_CURR"].values, folds=fl, P_SAMPLE=p_sample, twin_cols=twin_cols),
                open(CACHE, "wb"))
    print(f"[prep] {X.shape[1]} features (+{len(keys)} in-fold target encodings) in {time.time() - t0:.0f}s -> {CACHE}", flush=True)

def load_cache():
    global XALL, KEYS, y, n, N, TEST_IDX, test_ids, folds, P_SAMPLE, TWIN_COLS
    c = pickle.load(open(CACHE, "rb"))
    XALL, KEYS, y, n, N, test_ids, folds, P_SAMPLE = c["XALL"], c["KEYS"], c["y"], c["n"], c["N"], c["test_ids"], c["folds"], c["P_SAMPLE"]
    TEST_IDX = np.arange(n, N); TWIN_COLS = c.get("twin_cols")

# ----------------------------- leakage-free target encoding of the loan-product clusters -----------------------------
def _te_map(k, yy, prior):
    s = pd.DataFrame({"k": k, "y": yy}).groupby("k").y.agg(["sum", "count"])
    return (s["sum"] + prior * TE_ALPHA) / (s["count"] + TE_ALPHA)

def fold_te(tr, others, n_inner=5, seed=0):
    """Training rows get inner out-of-fold encodings; validation/test rows are encoded from all training rows."""
    prior = y[tr].mean(); TR = {}; OT = [dict() for _ in others]
    for nm, k in KEYS.items():
        oof = np.zeros(len(tr))
        for a, b in StratifiedKFold(n_inner, shuffle=True, random_state=seed).split(tr, y[tr]):
            m = _te_map(k[tr][a], y[tr][a], prior); oof[b] = pd.Series(k[tr][b]).map(m).fillna(prior).values
        TR[nm] = oof; m = _te_map(k[tr], y[tr], prior)
        for i, o in enumerate(others): OT[i][nm] = pd.Series(k[o]).map(m).fillna(prior).values
    return pd.DataFrame(TR), [pd.DataFrame(o) for o in OT]

def fold_matrices(tr, va, keep_cl=False):
    cols = [c for c in XALL.columns if keep_cl or c not in CL_COLS]
    TRte, (VAte, TEte) = fold_te(tr, [va, TEST_IDX])
    mk = lambda idx, te: pd.concat([XALL.iloc[idx][cols].reset_index(drop=True), te], axis=1)
    return mk(tr, TRte), mk(va, VAte), mk(TEST_IDX, TEte)

# ----------------------------- MODELS -----------------------------
HGB_PARAMS = {
    "hgb":   dict(learning_rate=0.04, max_iter=1500, max_leaf_nodes=12, l2_regularization=10.0, min_samples_leaf=200, max_features=0.3),
    "hgb_a": dict(learning_rate=0.04, max_iter=1200, max_leaf_nodes=24, l2_regularization=5.0,  min_samples_leaf=150, max_features=0.4),
    "hgb_f": dict(learning_rate=0.04, max_iter=430,  max_leaf_nodes=12, l2_regularization=10.0, min_samples_leaf=200, max_features=0.3),
}
def fit_hgb(name, tr, va, seed, gpu):
    Xtr, Xva, Xte = fold_matrices(tr, va)
    p = dict(HGB_PARAMS[name]); p["max_iter"] = min(p["max_iter"], MAX_ROUNDS)
    m = make_hgb(HistGradientBoostingClassifier, early_stopping=(name != "hgb_f"), validation_fraction=0.1, n_iter_no_change=50,
                 categorical_features=[Xtr.columns.get_loc(c) for c in CATS], random_state=seed, **p)
    m.fit(Xtr, y[tr])
    return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1], m.n_iter_

XGB_VARIANTS = {"xgb": {}, "xgb2": dict(max_depth=7, eta=0.03, min_child_weight=60, subsample=0.7, colsample_bytree=0.35, reg_lambda=10.0)}
def fit_xgb(name, tr, va, seed, gpu):
    import xgboost as xgb
    Xtr, Xva, Xte = fold_matrices(tr, va)
    p = dict(objective="binary:logistic", eval_metric="auc", eta=0.02, max_depth=5, min_child_weight=30, subsample=0.8,
             colsample_bytree=0.5, reg_lambda=5.0, reg_alpha=0.1, gamma=0.1, max_bin=256, seed=seed, nthread=THREADS)
    p.update(XGB_VARIANTS.get(name, {}))
    if int(xgb.__version__.split(".")[0]) >= 2:
        p.update(tree_method="hist", device=(f"cuda:{gpu}" if gpu is not None else "cpu"))
    else:
        p.update(tree_method="gpu_hist" if gpu is not None else "hist")
        if gpu is not None: p["gpu_id"] = gpu
    try:   # quantised matrices; with device=cuda the training data lives in GPU memory
        dtr = xgb.QuantileDMatrix(Xtr, y[tr], max_bin=256); dva = xgb.QuantileDMatrix(Xva, y[va], ref=dtr)
    except AttributeError:
        dtr = xgb.DMatrix(Xtr, y[tr]); dva = xgb.DMatrix(Xva, y[va])
    bst = xgb.train(p, dtr, num_boost_round=MAX_ROUNDS, evals=[(dva, "valid")], early_stopping_rounds=200, verbose_eval=False)
    it = (0, bst.best_iteration + 1)
    return bst.predict(xgb.DMatrix(Xva), iteration_range=it), bst.predict(xgb.DMatrix(Xte), iteration_range=it), bst.best_iteration

def fit_cat(name, tr, va, seed, gpu):
    from catboost import CatBoostClassifier, Pool
    Xtr, Xva, Xte = fold_matrices(tr, va, keep_cl=True)
    cc = CATS + CL_COLS
    for X_ in (Xtr, Xva, Xte):
        for c in cc: X_[c] = X_[c].astype("int32")
    prm = dict(iterations=MAX_ROUNDS, learning_rate=0.04, depth=6, l2_leaf_reg=10, loss_function="Logloss", eval_metric="AUC",
               random_seed=seed, verbose=0, od_type="Iter", od_wait=300, use_best_model=True, border_count=254,
               bootstrap_type="Bernoulli", subsample=0.8, allow_writing_files=False, thread_count=THREADS)
    if gpu is not None: prm.update(task_type="GPU", devices=str(gpu), gpu_ram_part=CAT_GPU_RAM_PART)   # leave memory for the other GPU models
    else: prm.update(task_type="CPU")
    m = CatBoostClassifier(**prm)
    m.fit(Pool(Xtr, y[tr], cat_features=cc), eval_set=Pool(Xva, y[va], cat_features=cc))
    return m.predict_proba(Pool(Xva, cat_features=cc))[:, 1], m.predict_proba(Pool(Xte, cat_features=cc))[:, 1], m.get_best_iteration()

def fit_lgb(name, tr, va, seed, gpu):
    import lightgbm as lgb
    Xtr, Xva, Xte = fold_matrices(tr, va)
    m = lgb.LGBMClassifier(n_estimators=MAX_ROUNDS, learning_rate=0.02, num_leaves=31, min_child_samples=100, subsample=0.8,
                           subsample_freq=1, colsample_bytree=0.5, reg_lambda=10.0, random_state=seed, n_jobs=THREADS, verbose=-1)
    m.fit(Xtr, y[tr], eval_set=[(Xva, y[va])], eval_metric="auc", callbacks=[lgb.early_stopping(200, verbose=False)])
    return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1], m.best_iteration_


# ----------------------------- linear / neural models: quantile + one-hot + spline features -----------------------------
def nn_matrices(Xtr, Xva, Xte):
    """Everything is fitted on the training rows of the fold only (median fill, quantile map, one-hot, splines)."""
    Xtr, Xva, Xte = (X.replace([np.inf, -np.inf], np.nan) for X in (Xtr, Xva, Xte))
    nc = [c for c in Xtr.columns if c not in CATS]; mc = [c for c in nc if Xtr[c].isnull().mean() > 0.01]; med = Xtr[nc].median()
    qt = QuantileTransformer(n_quantiles=200, output_distribution="normal", subsample=200000, random_state=0).fit(Xtr[nc].fillna(med))
    ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False).fit(Xtr[CATS])
    cont = [c for c in nc if Xtr[c].nunique() > 20]
    sp = SplineTransformer(n_knots=5, degree=3, knots="quantile", extrapolation="constant").fit(Xtr[cont].fillna(med[cont]))
    f = lambda X: np.hstack([qt.transform(X[nc].fillna(med)).astype(np.float32), X[mc].isnull().astype(np.float32).values,
                             ohe.transform(X[CATS]).astype(np.float32), sp.transform(X[cont].fillna(med[cont])).astype(np.float32)])
    return f(Xtr), f(Xva), f(Xte)

# name -> (hidden layers, learning rate, max_iter, patience, seed offset); "lr" = logistic regression
NN_SPECS = {"lr": None, "mlpA": ((128, 64), 1e-3, 40, 4, 101), "mlpB": ((128, 64), 1e-3, 40, 4, 102),
            "mlpC": ((256, 128), 5e-4, 80, 8, 103), "mlp3": ((256, 128, 64), 5e-4, 80, 8, 104)}
def fit_nn(name, tr, va, seed, gpu):
    Xtr, Xva, Xte = fold_matrices(tr, va); A, B, C = nn_matrices(Xtr, Xva, Xte); spec = NN_SPECS[name]
    if spec is None: m = LogisticRegression(C=0.02, max_iter=60 if SMOKE else 500)
    else:
        hid, lr, mi, nic, off = spec
        m = MLPClassifier(hidden_layer_sizes=hid, alpha=1e-2, batch_size=512, learning_rate_init=lr, max_iter=min(mi, 6) if SMOKE else mi,
                          early_stopping=True, validation_fraction=0.1, n_iter_no_change=nic, random_state=seed + off)
    m.fit(A, y[tr])
    return m.predict_proba(B)[:, 1], m.predict_proba(C)[:, 1], getattr(m, "n_iter_", 0)

FIT = {"xgb": fit_xgb, "xgb2": fit_xgb, "cat": fit_cat, "lgb": fit_lgb, "hgb": fit_hgb, "hgb_a": fit_hgb, "hgb_f": fit_hgb, **{k: fit_nn for k in NN_SPECS}}
GPU_MODELS = {"xgb", "xgb2", "cat"}
LIB = {"xgb": "xgboost", "xgb2": "xgboost", "cat": "catboost", "lgb": "lightgbm"}

def train_model(name):
    """One model, all folds, in this process. Finished folds are checkpointed and skipped on re-run."""
    if name not in FIT: sys.exit(f"unknown model {name}")
    if name in LIB:
        try: __import__(LIB[name])
        except ImportError: sys.exit(f"{name}: package '{LIB[name]}' is not installed")
    load_cache(); os.makedirs(PRED_DIR, exist_ok=True)
    gpu = 0 if (name in GPU_MODELS and N_GPUS > 0) else None
    if name in GPU_MODELS and gpu is None: print(f"[{name}] WARNING: no GPU detected, running on CPU (slow)", flush=True)
    print(f"[{name}] start: gpu={gpu} threads={THREADS} folds={len(folds)} seeds={N_SEEDS}", flush=True)
    for s in range(N_SEEDS):
        for f, (tr, va) in enumerate(folds):
            path = os.path.join(PRED_DIR, f"{name}__s{s}__f{f}.npz")
            if os.path.exists(path): print(f"[{name} fold {f + 1}/{len(folds)}] already done, skipping", flush=True); continue
            t0 = time.time(); pv, pt, it = FIT[name](name, tr, va, SEED + s, gpu)
            np.savez(path, va=pv, te=pt, va_idx=va, it=it)
            print(f"[{name} fold {f + 1}/{len(folds)} seed {s}] AUC={roc_auc_score(y[va], pv):.5f} iters={it} ({time.time() - t0:.0f}s)", flush=True)
    print(f"[{name}] DONE", flush=True)

# ----------------------------- BLEND + SUBMISSION -----------------------------
def load_model_preds(name):
    found = {}
    for fpath in glob.glob(os.path.join(PRED_DIR, f"{name}__s*__f*.npz")):
        m = re.match(rf"{re.escape(name)}__s(\d+)__f(\d+)\.npz", os.path.basename(fpath))
        if m: found.setdefault(int(m.group(1)), {})[int(m.group(2))] = fpath
    seeds = [s for s, fl in found.items() if len(fl) == len(folds)]          # complete seeds only
    if not seeds: return None
    oof = np.zeros(n); tep = np.zeros(len(TEST_IDX))
    for s in seeds:
        for f, fpath in found[s].items():
            z = np.load(fpath); oof[z["va_idx"]] += z["va"] / len(seeds); tep += z["te"] / (len(folds) * len(seeds))
    return oof, tep

def adversarial_weights():
    """Weights that make validation rows look like the test set: w = p(test|x) / p(train|x) from a train-vs-test classifier."""
    cols = [c for c in XALL.columns if c not in CL_COLS]; Xa = XALL[cols]
    ya = np.r_[np.zeros(n), np.ones(N - n)]; p = np.zeros(N); cidx = [Xa.columns.get_loc(c) for c in CATS]
    for a, b in StratifiedKFold(3, shuffle=True, random_state=0).split(Xa, ya):
        m = HistGradientBoostingClassifier(max_iter=30 if SMOKE else 120, learning_rate=0.1, categorical_features=cidx,
                                           early_stopping=True, random_state=0).fit(Xa.iloc[a], ya[a])
        p[b] = m.predict_proba(Xa.iloc[b])[:, 1]
    print(f"[adversarial] train-vs-test AUC = {roc_auc_score(ya, p):.4f}  (0.50 = identical distributions)", flush=True)
    pt = np.clip(p[:n], 1e-3, 1 - 1e-3); w = pt / (1 - pt) * (n / (N - n))
    return np.clip(w / w.mean(), 0.25, 4.0)

def blend():
    load_cache(); results = {}
    for m in MODELS:
        r = load_model_preds(m)
        if r is None: print(f"[blend] {m}: no complete predictions, skipped", flush=True)
        else: results[m] = r
    assert results, f"no model finished - check the logs in {LOG_DIR}"
    rk = lambda a: rankdata(a) / len(a)
    names = list(results); R_oof = np.column_stack([rk(results[k][0]) for k in names]); R_te = np.column_stack([rk(results[k][1]) for k in names])
    auc = lambda v: roc_auc_score(y, R_oof @ v)
    best_w = np.ones(len(names)) / len(names); best_auc = auc(best_w)          # start from equal weights, move 0.05 / 0.025 of weight while OOF AUC improves
    for step in (0.05, 0.025):
        improved = True
        while improved:
            improved = False
            for a_ in range(len(names)):
                for b_ in range(len(names)):
                    if a_ == b_ or best_w[a_] < step - 1e-9: continue
                    c = best_w.copy(); c[a_] -= step; c[b_] += step; v = auc(c)
                    if v > best_auc + 1e-7: best_w, best_auc, improved = c, v, True
    W_TL = adversarial_weights()
    print("\n================ RESULTS ================")
    print("  model   plain OOF AUC | test-like OOF AUC   (test-like = validation re-weighted to resemble the test set)")
    for k in names: print(f"  {k:6s}  {roc_auc_score(y, results[k][0]):.5f}       | {roc_auc_score(y, results[k][0], sample_weight=W_TL):.5f}")
    bo = R_oof @ best_w
    print(f"  blend   {roc_auc_score(y, bo):.5f}       | {roc_auc_score(y, bo, sample_weight=W_TL):.5f}    weights = { {k: round(float(v), 3) for k, v in zip(names, best_w) if v > 0} }")
    print("  Compare leaderboard scores with the test-like column; the leaderboard itself is noisy (+/-0.004 on 61.5k rows).")
    sub = pd.DataFrame({"SK_ID_CURR": test_ids, "TARGET": R_te @ best_w})
    if P_SAMPLE and not SMOKE:
        sample = pd.read_csv(P_SAMPLE); sub = sample[["SK_ID_CURR"]].merge(sub, on="SK_ID_CURR", how="left")
        assert len(sub) == len(sample) and sub.TARGET.notnull().all(), "submission does not match sample_submission"
    os.makedirs(OUT_DIR, exist_ok=True)
    out = os.path.join(OUT_DIR, "submission_smoke.csv" if SMOKE else "submission.csv"); sub.to_csv(out, index=False)
    pickle.dump({"oof": {k: v[0] for k, v in results.items()}, "test": {k: v[1] for k, v in results.items()}, "y": y,
                 "w_testlike": W_TL, "test_ids": test_ids}, open(os.path.join(WORK_DIR, f"oof_test_preds{TAG}.pkl"), "wb"))
    print(f"\n[done] wrote {out}  shape={sub.shape}  range=({sub.TARGET.min():.4f}, {sub.TARGET.max():.4f})", flush=True)
    if os.environ.get("TWINS", "1") == "1" and TWIN_COLUMNS is not None and TWIN_COLS is not None:
        try:
            p_tw, gain = twin_adjust(bo, R_te @ best_w, y, TWIN_COLS, n)
            sub2 = pd.DataFrame({"SK_ID_CURR": test_ids, "TARGET": p_tw})
            if P_SAMPLE and not SMOKE:
                sample = pd.read_csv(P_SAMPLE); sub2 = sample[["SK_ID_CURR"]].merge(sub2, on="SK_ID_CURR", how="left"); assert sub2.TARGET.notnull().all() and len(sub2) == len(sample)
            out2 = os.path.join(OUT_DIR, "submission_twins_smoke.csv" if SMOKE else "submission_twins.csv"); sub2.to_csv(out2, index=False)
            print(f"[done] wrote {out2}  <- THE RECOMMENDED SUBMISSION (blend + twin adjustment, probabilities)", flush=True)
        except Exception:
            print("[twins] adjustment failed, the plain blend above is still valid:\n" + traceback.format_exc(), flush=True)


# ----------------------------- TWIN ADJUSTMENT: the same person appearing under different IDs -----------------------------
# Application dates differ, so every DAYS_* field is shifted by the same amount for the same person. Differences such as
# DAYS_REGISTRATION - DAYS_BIRTH therefore stay constant and act as a fingerprint. Among train pairs, the second application
# defaults 41.5% of the time if the first defaulted and 3.5% if it did not - and the direction matters: if the EARLIER application defaulted, the later loan
# defaults ~70% of the time, if the LATER one defaulted the earlier loan defaults ~23%. A second stage uses these outcomes (plus time gap, contract type, change in credit/income).
TWIN_COLUMNS = ["DAYS_BIRTH", "DAYS_ID_PUBLISH", "DAYS_REGISTRATION", "DAYS_EMPLOYED", "REGION_POPULATION_RELATIVE", "CODE_GENDER", "NAME_CONTRACT_TYPE",
                "AMT_CREDIT", "AMT_INCOME_TOTAL", "AMT_GOODS_PRICE", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
def twin_adjust(oof, te, yy, cols, nn, verbose=True):
    """oof / te: blended scores for train (out-of-fold) and test rows. cols: TWIN_COLUMNS for all rows (train first, then test).
    Returns calibrated default probabilities for the test rows with the twin adjustment applied (plus the group-CV AUC gain)."""
    NN = len(cols); f = cols.reset_index(drop=True).copy(); rk_ = lambda a: rankdata(a) / len(a)
    f["A"] = f.DAYS_REGISTRATION - f.DAYS_BIRTH; f["B"] = f.DAYS_ID_PUBLISH - f.DAYS_BIRTH
    f["C"] = f.DAYS_EMPLOYED.where(f.DAYS_EMPLOYED != 365243) - f.DAYS_BIRTH; f["R"] = f.REGION_POPULATION_RELATIVE; g = "CODE_GENDER"
    levels = {"same_day": ["DAYS_BIRTH", "DAYS_ID_PUBLISH", "DAYS_REGISTRATION"], "ABgCR": ["A", "B", g, "C", "R"], "ACgR": ["A", "C", g, "R"],
              "ABgR": ["A", "B", g, "R"], "BCgR": ["B", "C", g, "R"]}
    rows, cols_ = [], []
    for keys in levels.values():                      # link rows that share a fingerprint -> connected components = persons
        ok = f[keys].notnull().all(axis=1).values; idx = np.where(ok)[0]
        if len(idx) < 2: continue
        gid = f.loc[ok, keys].groupby(keys).ngroup().values; order = idx[np.argsort(gid, kind="stable")]; gg = np.sort(gid); same = gg[1:] == gg[:-1]
        rows += list(order[:-1][same]); cols_ += list(order[1:][same])
    pid = connected_components(coo_matrix((np.ones(len(rows)), (rows, cols_)), shape=(NN, NN)), directed=False)[1] if rows else np.arange(NN)
    z_tr = norm.ppf(np.clip(rk_(oof), 1e-4, 1 - 1e-4)); z_te = norm.ppf(np.clip(rk_(te), 1e-4, 1 - 1e-4)); z_all = np.r_[z_tr, z_te]
    cal = LogisticRegression(C=100).fit(z_tr.reshape(-1, 1), yy); p_tr = cal.predict_proba(z_tr.reshape(-1, 1))[:, 1]; p_te = cal.predict_proba(z_te.reshape(-1, 1))[:, 1]
    ylab = np.r_[yy.astype(float), np.full(NN - nn, np.nan)]; istr = np.r_[np.ones(nn, bool), np.zeros(NN - nn, bool)]
    DB = f.DAYS_BIRTH.values; CT = (f.NAME_CONTRACT_TYPE == "Cash loans").values
    LC, LI, LG, E2 = np.log1p(f.AMT_CREDIT.values), np.log1p(f.AMT_INCOME_TOTAL.values), np.log1p(f.AMT_GOODS_PRICE.values), f.EXT_SOURCE_2.values
    E1, E3 = f.EXT_SOURCE_1.values, f.EXT_SOURCE_3.values; m1, m3 = np.nanmean(E1), np.nanmean(E3)
    names = ["def1", "ok_only", "def2", "only_unlab", "zmean_other", "gap_def", "same_ct_def",              # outcomes of labelled twins
             "later", "dcredit", "dincome", "small_gap", "dext2", "same_goods", "n_twins",                 # how this application differs from its nearest twin
             "def_before", "def_after", "ok_before", "ok_after", "is_first", "is_last",                   # direction: DAYS_BIRTH closer to 0 = EARLIER application
             "ext1_twin", "ext3_twin", "ct_self", "ct_twin", "gap_before", "gap_after", "z_before", "z_after", "has_before", "has_after"]   # extras (used by the ensemble only)
    F = {k: np.zeros(NN) for k in names}
    sizes = pd.Series(pid).value_counts(); multi = set(sizes[sizes >= 2].index)
    for p_, ix in pd.Series(np.arange(NN)).groupby(pid).__iter__():
        if p_ not in multi: continue
        ix = ix.values
        for i in ix:
            oth = [j for j in ix if j != i]; lab = [j for j in oth if istr[j]]
            F["zmean_other"][i] = np.mean([z_all[j] for j in oth]); F["only_unlab"][i] = float(len(lab) == 0); F["n_twins"][i] = len(oth)
            gaps_all = np.array([abs(DB[i] - DB[j]) for j in oth]); jn = oth[int(np.argmin(gaps_all))]          # nearest twin in time (labelled or not)
            F["is_first"][i] = float(all(DB[i] >= DB[j] for j in oth)); F["is_last"][i] = float(all(DB[i] <= DB[j] for j in oth)); F["later"][i] = float(DB[i] < DB[jn])
            F["dcredit"][i] = LC[i] - LC[jn]; F["dincome"][i] = LI[i] - LI[jn]; F["small_gap"][i] = float(gaps_all.min() <= 7); F["same_goods"][i] = float(LG[i] == LG[jn])
            F["dext2"][i] = 0.0 if (np.isnan(E2[i]) or np.isnan(E2[jn])) else E2[i] - E2[jn]
            for arr, mm, nm_ in ((E1, m1, "ext1_twin"), (E3, m3, "ext3_twin")):          # borrow the twin's external score where this application's is blank
                if np.isnan(arr[i]):
                    cand = [j for j in oth if not np.isnan(arr[j])]
                    if cand: jj = min(cand, key=lambda j: abs(DB[i] - DB[j])); F[nm_][i] = arr[jj] - mm
            F["ct_self"][i] = float(CT[i]); F["ct_twin"][i] = float(CT[jn])
            zb_ = [z_all[j] for j in oth if DB[j] > DB[i]]; za_ = [z_all[j] for j in oth if DB[j] < DB[i]]
            F["has_before"][i] = float(len(zb_) > 0); F["has_after"][i] = float(len(za_) > 0); F["z_before"][i] = np.mean(zb_) if zb_ else 0.0; F["z_after"][i] = np.mean(za_) if za_ else 0.0
            if lab:
                gb_ = [abs(DB[i] - DB[j]) for j in lab if DB[j] > DB[i] and ylab[j] == 1]; ga_ = [abs(DB[i] - DB[j]) for j in lab if DB[j] < DB[i] and ylab[j] == 1]
                F["gap_before"][i] = np.log1p(min(gb_)) if gb_ else 0.0; F["gap_after"][i] = np.log1p(min(ga_)) if ga_ else 0.0
                yb = [ylab[j] for j in lab if DB[j] > DB[i]]; ya_ = [ylab[j] for j in lab if DB[j] < DB[i]]
                F["def_before"][i] = float(sum(yb) >= 1); F["ok_before"][i] = float(len(yb) > 0 and sum(yb) == 0)
                F["def_after"][i] = float(sum(ya_) >= 1); F["ok_after"][i] = float(len(ya_) > 0 and sum(ya_) == 0)
                yl = np.array([ylab[j] for j in lab]); gaps = np.array([abs(DB[i] - DB[j]) for j in lab])
                F["def1"][i] = float(yl.sum() >= 1); F["def2"][i] = float(yl.sum() >= 2); F["ok_only"][i] = float(yl.sum() == 0)
                if yl.sum() >= 1:
                    k = int(np.argmin(np.where(yl == 1, gaps, 1e9))); F["gap_def"][i] = np.log1p(gaps[k]); F["same_ct_def"][i] = float(CT[i] == CT[lab[k]])
    has_twin = pd.Series(pid).map(sizes).values >= 2
    tw_tr = np.where(has_twin[:nn])[0]; tw_te = np.where(has_twin[nn:])[0]
    base_auc = roc_auc_score(yy, p_tr); gain = 0.0
    if len(tw_tr) < 200 or yy[tw_tr].sum() < 20:
        if verbose: print(f"[twins] only {len(tw_tr)} train rows have a twin - adjustment skipped", flush=True)
        return p_te, 0.0
    K2 = names[:20]                                                    # the v10 feature set; names[20:] are the extras
    variants = [("LR, v10 features", K2, lambda: LogisticRegression(C=10, max_iter=2000)),
                ("LR, all features, stronger regularisation", names, lambda: LogisticRegression(C=3, max_iter=3000)),
                ("small gradient-boosted stage, all features", names, lambda: HistGradientBoostingClassifier(max_depth=2, max_iter=80, learning_rate=0.05, min_samples_leaf=40, l2_regularization=5.0, random_state=0))]
    design = lambda cols_, z_, idx_, off: np.column_stack([z_[idx_]] + [F[k][off + idx_] for k in cols_])
    adjs = []
    for nm_, cols_, mk in variants:
        X_ = design(cols_, z_tr, tw_tr, 0); a_ = p_tr.copy()
        for a, b in GroupKFold(5).split(X_, yy[tw_tr], groups=pid[:nn][tw_tr]):
            a_[tw_tr[b]] = mk().fit(X_[a], yy[tw_tr][a]).predict_proba(X_[b])[:, 1]
        adjs.append(a_)
        if verbose: print(f"[twins]   {nm_:45s} group-CV AUC {roc_auc_score(yy, a_):.5f} (gain {roc_auc_score(yy, a_) - base_auc:+.5f})", flush=True)
    adj = np.mean(adjs, axis=0); gain = roc_auc_score(yy, adj) - base_auc
    if len(tw_te):
        p_te[tw_te] = np.mean([mk().fit(design(cols_, z_tr, tw_tr, 0), yy[tw_tr]).predict_proba(design(cols_, z_te, tw_te, nn))[:, 1] for _, cols_, mk in variants], axis=0)
    if verbose:
        print(f"[twins] persons found on several rows: {len(multi)} | train rows with a twin: {len(tw_tr)} | test rows with a twin: {len(tw_te)}", flush=True)
        print(f"[twins] ENSEMBLE of the three stages: group-CV AUC base {base_auc:.5f} -> {base_auc + gain:.5f}  (gain {gain:+.5f})", flush=True)
    return p_te, gain

# ----------------------------- ORCHESTRATOR: several models at once on the one GPU -----------------------------
def _last_line(path):
    try:
        with open(path, "rb") as fh:
            fh.seek(0, 2); size = fh.tell(); fh.seek(max(0, size - 400)); tail = fh.read().decode(errors="ignore")
        lines = [l.strip() for l in tail.splitlines() if l.strip()]
        return lines[-1][:120] if lines else "(starting)"
    except Exception:
        return "(no log yet)"

def run_all():
    t_start = time.time()
    for d_ in (WORK_DIR, PRED_DIR, LOG_DIR): os.makedirs(d_, exist_ok=True)
    if os.path.exists(CACHE) and os.environ.get("REUSE_CACHE", "1") == "1": print(f"[prep] reusing {CACHE} (set REUSE_CACHE=0 to rebuild)", flush=True)
    else: prep()
    models = [m for m in MODELS if m in FIT]
    n_gpu = sum(m in GPU_MODELS for m in models); n_cpu = len(models) - n_gpu; cpu = os.cpu_count() or 2
    cpu_threads = max(1, (cpu - n_gpu) // max(1, n_cpu)); gpu_threads = max(1, min(2, cpu // 2))
    print(f"[run] GPUs={N_GPUS} models={models} up to {MAX_PARALLEL} at once; CPU threads per CPU model={cpu_threads}", flush=True)
    queue = list(models); running = {}; rc = {}; last_print = 0.0
    while queue or running:
        while queue and len(running) < MAX_PARALLEL:
            m = queue.pop(0); th = gpu_threads if m in GPU_MODELS else cpu_threads
            env = dict(os.environ, HC_THREADS=str(th), OMP_NUM_THREADS=str(th), PYTHONUNBUFFERED="1")
            log = open(os.path.join(LOG_DIR, f"{m}.log"), "a")
            running[m] = (subprocess.Popen([sys.executable, os.path.abspath(__file__), "train", m], stdout=log, stderr=subprocess.STDOUT, env=env), log)
            print(f"[run] started {m}", flush=True); time.sleep(3)          # stagger so the processes do not hit the GPU at the same instant
        for m, (p, log) in list(running.items()):
            if p.poll() is not None:
                log.close(); rc[m] = p.returncode; del running[m]
                print(f"[run] {m} finished (exit code {rc[m]}) at {time.time() - t_start:.0f}s", flush=True)
        if running and time.time() - last_print > 30:
            last_print = time.time(); print(f"[status {time.time() - t_start:.0f}s]", flush=True)
            for m in running: print(f"    {m:6s} {_last_line(os.path.join(LOG_DIR, f'{m}.log'))}", flush=True)
        time.sleep(2)
    failed = [m for m, c in rc.items() if c != 0]
    if failed: print(f"[run] models that FAILED (see {LOG_DIR}/<model>.log): {failed} - blending the rest", flush=True)
    blend()
    print(f"[total] {time.time() - t_start:.0f}s", flush=True)

if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "all"
    if mode == "prep": prep()
    elif mode == "train": train_model(sys.argv[2])
    elif mode == "blend": blend()
    elif mode == "all": run_all()
    else: sys.exit("usage: python home_credit_colab.py [all | prep | train <model> | blend]")


In [ ]:
import os
os.environ.update(
    N_FOLDS='10',        # every model trains on 90% of the data in each fold
    N_SEEDS='1',         # raise to 2-3 for a little extra stability (longer run)
    MODELS='xgb,xgb2,cat,hgb_f,lr,mlpA,mlpB,mlp3',
    MAX_PARALLEL='4',    # models training simultaneously on the one T4 / the CPU cores
    TWINS='1',           # twin adjustment as the last stage (set '0' to skip)
)
# os.environ['WORK_DIR'] = '/content/drive/MyDrive/hc_work'   # uncomment to keep checkpoints on Drive (survives a disconnect)
# os.environ['SMOKE'] = '1'   # uncomment for a ~5 minute test on a small sample first (writes *_smoke.csv only)
!python home_credit_all_in_one.py all

In [ ]:
# OPTIONAL: rank-blend the twin-adjusted result with another submission file you already have (any scale - ranks are used).
# Do not assume this helps: only keep it if the leaderboard says so. Leave EXTERNAL empty to skip.
import pandas as pd
from scipy.stats import rankdata
EXTERNAL = {}     # e.g. {'/content/my_other_submission.csv': 0.3}   -> 30% weight for that file
if EXTERNAL:
    mine = pd.read_csv('/content/submission_twins.csv'); out = rankdata(mine.TARGET) / len(mine) * (1 - sum(EXTERNAL.values()))
    for path, w in EXTERNAL.items():
        o = mine[['SK_ID_CURR']].merge(pd.read_csv(path)[['SK_ID_CURR', 'TARGET']], on='SK_ID_CURR', how='left')
        assert o.TARGET.notnull().all(), 'ids do not match'; out = out + w * rankdata(o.TARGET) / len(o)
    mine['TARGET'] = out / out.max(); mine.to_csv('/content/submission_blended.csv', index=False); print('wrote /content/submission_blended.csv')

In [ ]:
# download the file to submit
from google.colab import files
files.download('/content/submission_twins.csv')